In [ ]:
import pandas as pd
import spacy
from difflib import get_close_matches

# Charger le modèle spaCy (par exemple, le modèle français ou anglais)
nlp = spacy.load("fr_core_news_sm")  # Remplacez par "en_core_web_sm" pour l'anglais

# Charger le dataset des noms de véhicules corrects
correct_vehicles_path = "cars2027.csv"  # Chemin vers le fichier des noms corrects
df_correct = pd.read_csv(correct_vehicles_path)
correct_vehicle_list = df_correct["Nom_Véhicule"].tolist()

# Charger le dataset des noms d'entrée (mal orthographiés)
input_vehicles_path = "cars_error3.csv"  # Chemin vers le fichier des noms d'entrée
df_input = pd.read_csv(input_vehicles_path)

# Fonction pour corriger un nom de véhicule
def correct_vehicle_name(input_name, vehicle_list):
    # Utiliser spaCy pour tokeniser et normaliser le texte
    doc = nlp(input_name)
    normalized_input = " ".join([token.text for token in doc])
    

    # Convertir toutes les chaînes en minuscules avant comparaison
    vehicle_list1 = [s.lower() for s in vehicle_list]  # Convertir en minuscules
    normalized_input1 = normalized_input.lower()
    
    
    # Calculer la similarité manuellement
    similarities = [(s, difflib.SequenceMatcher(None, normalized_input1, s).ratio()) for s in vehicle_list1]

    # Filtrer avec un seuil de 0.8
    matches = [s for s, score in similarities if score >= 0.85]
    
    matches1 = [vehicle_list[vehicle_list1.index(r)] for r in matches]  
    
    if matches1:
        return matches1[0]  # Retourne la meilleure correspondance
    else:
        return "Nom de véhicule non reconnu"

# Appliquer la correction à chaque entrée du dataset d'entrée
df_input["corrected_name"] = df_input["Nom_Véhicule"].apply(
    lambda x: correct_vehicle_name(x, correct_vehicle_list)
)

# Sauvegarder les résultats dans un nouveau fichier CSV
output_path = "corrected_vehicles.csv"
df_input.to_csv(output_path, index=False)

print(f"Correction terminée. Résultats sauvegardés dans {output_path}")